# HookLab MIE Recognition v0.3.1 — ruta móvil

Esta libreta permite ejecutar desde el celular la cadena candidata `audio → M + H + T → reconocimiento`. v0.3.1 conserva las observaciones crudas y añade capas derivadas para recuperar huecos melódicos, resolver el tactus y alinear armonía a tiempos fuertes confirmados. La salida permanece en `D0_EXPLORATORY`, con `scientific_d_unlocked=false`.

In [ ]:
# PASO 1 — Instalar HookLab en Python 3.10 aislado (compatible con Basic Pitch)
import os, pathlib, subprocess, sys, time
ROOT = pathlib.Path('/content/hooklab-time')
VENV = pathlib.Path('/content/mie-py310')
REF = 'codex/mie-recovery-v0.3'
def run(label, command, timeout=600, env=None):
    print(f'▶ {label}', flush=True)
    started = time.time()
    subprocess.run(command, check=True, timeout=timeout, env=env)
    print(f'✓ {label} · {time.time()-started:.0f} s', flush=True)
print('Kernel de Colab:', sys.version.split()[0], '· MIE usará Python 3.10 aislado', flush=True)
if not ROOT.exists():
    run('Descargar HookLab', ['git','clone','--depth','1','--branch',REF,'https://github.com/basspauloandres-svg/hooklab-time.git',str(ROOT)], 180)
else:
    run('Actualizar HookLab', ['git','-C',str(ROOT),'fetch','--depth','1','origin',REF], 180)
    run('Sincronizar revisión', ['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'], 60)
os.chdir(ROOT)
run('Preparar audio del sistema', ['apt-get','update','-qq'], 180)
run('Instalar FFmpeg', ['apt-get','install','-y','-qq','ffmpeg'], 180)
run('Instalar gestor de entorno', [sys.executable,'-m','pip','install','--disable-pip-version-check','uv==0.8.17'], 300)
run('Crear Python 3.10 aislado', [sys.executable,'-m','uv','venv',str(VENV),'--python','3.10','--seed'], 300)
PY = VENV/'bin/python'
run('Instalar PyTorch CPU compatible', [str(PY),'-m','pip','install','--disable-pip-version-check','--progress-bar','off','torch==2.2.2+cpu','torchaudio==2.2.2+cpu','--extra-index-url','https://download.pytorch.org/whl/cpu'], 600)
run('Instalar motores MIE fijados', [str(PY),'-m','pip','install','--disable-pip-version-check','--progress-bar','off','-r','mie_core/requirements-colab-py310.txt'], 600)
run('Verificar motores', [str(PY),'-c','import torch, torchaudio, demucs, librosa, onnxruntime, soundfile; from basic_pitch.inference import predict; print("✓ imports efectivos M+H+T")'], 120)
print('INSTALACIÓN LISTA · rama', REF)

In [ ]:
# PASO 2 — Cargar un audio autorizado desde el celular
from google.colab import files
import hashlib, pathlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Carga exactamente un archivo de audio')
original_name, audio_bytes = next(iter(uploaded.items()))
INPUT = pathlib.Path('/content/mie_mobile_input')
INPUT.write_bytes(audio_bytes)
SOURCE_SHA256 = hashlib.sha256(audio_bytes).hexdigest()
print('AUDIO LISTO ·', original_name, '· SHA-256', SOURCE_SHA256)

In [ ]:
# PASO 3 — Ejecutar separación, melodía, armonía, beat, razonamiento y resíntesis
import json, os, pathlib, shutil, subprocess, time
WORK = pathlib.Path('/content/mie_mobile_work')
if WORK.exists(): shutil.rmtree(WORK)
(WORK/'result').mkdir(parents=True)
wav = WORK/'source.wav'
run('Convertir audio', ['ffmpeg','-y','-i',str(INPUT),'-ar','44100','-ac','2',str(wav)], 300)
run('Separar voz, bajo, batería y acompañamiento', [str(PY),'-m','demucs','-n','htdemucs','-o',str(WORK/'stems'),str(wav)], 1200)
run('Medir melodía, armonía y beat', [str(PY),str(ROOT/'mie_core/run_mie_core.py'),'--audio',str(wav),'--stems',str(WORK/'stems'),'--output',str(WORK/'result')], 1200)
recovery_code = r'''
import json, pathlib, sys, time
from mie_core.mie_octave_plane_resolver import resolve_event_octaves
from mie_core.mie_recovery_pipeline import apply_reasoning
from mie_core.mie_recognition_contract import normalize
from mie_core.mie_recovery_resynthesis import render
work=pathlib.Path(sys.argv[1]); source_sha=sys.argv[2]; source_name=sys.argv[3]
raw=json.loads((work/'result/MIE_CORE_MHT_v0_2.json').read_text())
raw['notes']=resolve_event_octaves(raw.get('notes', []))
analysis_id='HL-COLAB-'+str(int(time.time()))
reasoned=apply_reasoning(raw, analysis_id=analysis_id)
output_wav=work/'result/MIE_RECOGNITION_MHT_v0_3_1.wav'
resynthesis=render(reasoned, output_wav)
result=normalize(reasoned, session_id=analysis_id, reference_sha256=source_sha, sensor_version='MIE_CORE_v0.3.1+RECOVERY_v0.3', ai_provenance=reasoned.get('ai_provenance'))
if result['status'] != 'PASS': raise RuntimeError(result)
result['resynthesis']=resynthesis; result['source_name']=source_name
(work/'result/MIE_RECOGNITION_v0_3_1.json').write_text(json.dumps(result,indent=2,ensure_ascii=False))
'''
env = dict(os.environ, PYTHONPATH=str(ROOT))
run('Aplicar recuperación y contrato', [str(PY),'-c',recovery_code,str(WORK),SOURCE_SHA256,original_name], 300, env)
result_path = WORK/'result/MIE_RECOGNITION_v0_3_1.json'
result = json.loads(result_path.read_text())
output_wav = WORK/'result/MIE_RECOGNITION_MHT_v0_3_1.wav'
audit = result['recognition']
print('RECONOCIMIENTO v0.3.1 LISTO · M',len(result['transcription']['melody_events']),'· recuperadas',audit['melody_recovery']['recovered_candidate_count'],'· H',len(result['transcription']['harmony_states']),'· T',len(result['transcription']['beat_events']),'· métrica',audit['tactus_resolution']['metric_resolution']['state'])

In [ ]:
# PASO 4 — Comparar entrada y reconstrucción; descargar resultados
from IPython.display import Audio, display
from google.colab import files
import shutil
print('A · AUDIO DE ENTRADA')
display(Audio(str(wav)))
print('B · RECONSTRUCCIÓN MIE v0.3.1')
display(Audio(str(output_wav)))
package = shutil.make_archive('/content/MIE_RECOGNITION_v0_3_1','zip',WORK/'result')
print('Pulsa aceptar cuando el teléfono solicite descargar el ZIP.')
files.download(package)

## Evaluación del productor

Escucha la reconstrucción y registra por separado: continuidad y octava de la melodía, correspondencia de la armonía, estabilidad del pulso y reconocimiento global de la obra. Una valoración auditiva favorable conserva el artefacto como candidato; la promoción científica requiere calibración independiente.